In [2]:
from pyvistra.io import load_image, save_tiff
import numpy as np
import focalstackutils as F
import matplotlib.pyplot as plt
from scipy.ndimage import map_coordinates
from pathlib import Path

In [7]:
def process_file(p: Path, offset_um = 0.7):
    img, meta = load_image(str(p))
    img3d = img[0, :, 0, :, :] # first channel is brightfield
    dapi3d = img[0, :, 1, :, :] # second channel is DAPI
    projected_dapi = np.sum(dapi3d, axis=0).astype(np.uint16)
    Nz, Ny, Nx = img3d.shape
    dz, dy, dx = meta['scale']
    grids, tile_info = F.find_focus_index_map(img3d)
    focus_indices = F.fit_focal_indices_to_poly2d(grids, Ny, Nx, tile_info)
    focal_slice = F.resample_focal_slice(img3d, focus_indices, dz, offset_um)
    meta = {"scale": (dz, dy, dx)}
    return focal_slice, projected_dapi, meta

In [8]:
sample_name = "wild-type"

ipdir = Path("/media/starrluxton/DE_extHD01/Bri/2026-07-20")
opdir = Path(f"/home/starrluxton/BurgessLab/yeast_monolayer_project/yeast_images/{sample_name}")

outdir_focal_slices = opdir / "focal_slices"
outdir_sum_projections = opdir / "projected_DAPI"

if not outdir_focal_slices.exists():
    outdir_focal_slices.mkdir(parents=True, exist_ok=True)
if not outdir_sum_projections.exists():
    outdir_sum_projections.mkdir(parents=True, exist_ok=True)

flist = list(f for f in ipdir.glob("*.ims"))

for fn in flist:
    print(f"Processing file: \n{fn.name} ...")
    fn_out = f"{fn.stem}.tiff"
    focal_slice, projected_dapi, meta = process_file(fn)
    print(f"    projected DAPI, min = {projected_dapi.min():.1f}, max = {projected_dapi.max():.1f}")
    
    save_tiff(outdir_sum_projections / fn_out, projected_dapi, scale=meta["scale"], input_axes="YX")
    save_tiff(outdir_focal_slices / fn_out, focal_slice, scale=meta["scale"], input_axes="YX")

Processing file: 
2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F0.ims ...
    projected DAPI, min = 5189.0, max = 54970.0
Processing file: 
2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F3.ims ...
    projected DAPI, min = 5706.0, max = 59758.0
Processing file: 
2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F4.ims ...
    projected DAPI, min = 141.0, max = 65495.0
Processing file: 
2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F2.ims ...
    projected DAPI, min = 3831.0, max = 33419.0
Processing file: 
2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F5.ims ...
    projected DAPI, min = 4103.0, max = 54395.0
Processing file: 
2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F6.ims ...
    projected DAPI, min = 3916.0, max = 53051.0
Processing file: 
2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F1.ims ...
    projected DAPI, min = 4493.0, max = 58365.0
